# EQB alpha/gamma grid search
Run equilibrium searches over a grid of (Lx, Lz) values using the shear-band guess strategy.
Stores unique equilibria and summary metrics (I, D, ||x||).

In [1]:
using CloudAtlas
using LinearAlgebra
using Random
using Statistics
using DelimitedFiles
using Printf

## Experiment parameters

In [ ]:
# Discretization
J, K, L = 1, 3, 5

# Reynolds number
Re = 300.0

# Symmetry group (edit as needed)
sx, sy, sz, tx, tz = halfbox_symmetries()
H = [sx*sy*sz]

# Grid in physical box sizes
Lx_vals = range(2π, 4π; length=5)
Lz_vals = range(π, 2π; length=5)

# Number of guesses per grid point
N = 50

# Guess strategy
guess_strategy = :shear_band
shear_min, shear_max = 1.0, 3.0
rest_scale = 0.1
frac = 0.9        # fraction of |shear| mass for dominant modes
top_k = nothing   # alternatively choose a fixed number of dominant modes

# Dedup tolerances
fp_tol = (cx = 1e-3, cz = 1e-3, nm = 2e-2, shear = 2e-2)

# Solver parameters
hookparams = SearchParams(ftol=1e-8, xtol=1e-10, Nnewton=25, Nhook=6, δ=0.02, verbosity=0)

# Output
out_dir = joinpath(@__DIR__, "eqb_alpha_gamma_grid")
mkpath(out_dir)

## Helpers

In [ ]:
function solve_eqb(model, Re, xguess, hookparams)
    f = x -> model.f(x, Re)
    Df = x -> model.Df(x, Re)
    return hookstepsolve(f, Df, xguess, hookparams)
end

function summarize_solution(model, Dmat, x, Re)
    res = norm(model.f(x, Re)) / max(norm(x), eps(eltype(x)))
    I = power_input(model, x)
    D = dissipation_rate(Dmat, x)
    return (norm = norm(x), shear = I, dissipation = D, res = res)
end

## Grid search

In [ ]:
results = NamedTuple[]
rng = Random.default_rng()

for Lx in Lx_vals
    for Lz in Lz_vals
        α = 2π / Lx
        γ = 2π / Lz
        println("
=== Lx=$(Lx), Lz=$(Lz) (α=$(α), γ=$(γ)) ===")

        model = ODEModel(α, γ, J, K, L, H)
        Dmat = build_dissipation_matrix(model)

        sols = Vector{Vector{Float64}}()
        fps = Vector{SolutionFingerprint}()

        for attempt in 1:N
            ξ_guess = build_guess(
                model, rng;
                strategy = guess_strategy,
                shear_min = shear_min,
                shear_max = shear_max,
                rest_scale = rest_scale,
                frac = frac,
                top_k = top_k,
            )
            x_guess, _, _ = extract_components(ξ_guess, model)

            x_sol, converged = solve_eqb(model, Re, x_guess, hookparams)
            if converged
                fp = fingerprint(model, x_sol)
                if is_distinct(fp, fps; tol = fp_tol)
                    push!(fps, fp)
                    push!(sols, x_sol)
                    println("  + unique eqb: ||x||=$(round(norm(x_sol), digits=4)), shear=$(round(fp.shear, digits=4))")
                end
            end
        end

        # Save solutions and summary rows
        for (i, x_sol) in enumerate(sols)
            fname = @sprintf("eqb_Lx%.4f_Lz%.4f_id%03d.asc", Lx, Lz, i)
            save(x_sol, joinpath(out_dir, fname))

            summ = summarize_solution(model, Dmat, x_sol, Re)
            push!(results, (
                Lx = Lx, Lz = Lz, α = α, γ = γ, id = i,
                norm = summ.norm, shear = summ.shear, dissipation = summ.dissipation, res = summ.res
            ))
        end

        println("Found $(length(sols)) unique equilibria.")
    end
end

## Save summary CSV

In [ ]:
summary_path = joinpath(out_dir, "summary.csv")
open(summary_path, "w") do io
    println(io, "Lx,Lz,alpha,gamma,id,norm,shear,dissipation,residual")
    for r in results
        println(io, "$(r.Lx),$(r.Lz),$(r.α),$(r.γ),$(r.id),$(r.norm),$(r.shear),$(r.dissipation),$(r.res)")
    end
end
println("Wrote summary to $summary_path")

## Heatmap of unique equilibria
Counts of unique solutions per (Lx, Lz).

In [ ]:
using CairoMakie

# Build a matrix of counts aligned with Lx_vals/Lz_vals
count_mat = zeros(Int, length(Lx_vals), length(Lz_vals))
for r in results
    i = findfirst(==(r.Lx), Lx_vals)
    j = findfirst(==(r.Lz), Lz_vals)
    count_mat[i, j] += 1
end

fig = Figure(size = (700, 500))
ax = Axis(fig[1, 1], xlabel = "Lz", ylabel = "Lx", title = "Unique equilibria count")
hm = heatmap!(ax, Lz_vals, Lx_vals, count_mat; colormap = :viridis)
Colorbar(fig[1, 2], hm, label = "count")
display(fig)